# Intrinsic curvature patch-output parity

This one-off check compares **every output element** of the original NumPy `scalar_curvature` wrapper with the current `_scalar_curvature_in_patches`, using the **same input array** and the **same unchanged intrinsic `scalar_curvature_batch`**. It checks shape, dtype, and each element's bytes, so it also catches signed-zero and NaN-payload differences. It does not compare intrinsic with extrinsic curvature.

The function in the next code cell is copied verbatim from `microscope/computations_grid/curvature.py` at inner commit `35a1641e3267344c042a624af63a33b47ff226a8`. Its SHA-256 as displayed here, without a trailing newline, is `405aa62b6f008b5f846c008871d74082ba60500cfd824e5a67c9eb86ea7c048a`. The function body is intentionally kept as historical reference code. No package code is changed by this notebook.

The default run covers regular and irregular 2D ellipsoids and the 2D hyperboloid. Set `RUN_SLOW_3D = True` for the 3D ellipsoid and hyperboloid cases from `test_curvature.py`; set `RUN_GIANT_TOTAL_GRID = True` to include the two 10,223 × 10,344 ellipsoid grids. The high-ambient 3D cases and giant grid may require substantial RAM and time. The notebook stores no generated outputs.

In [1]:
import numpy as np
import torch

from microscope.patches import extract_patches, stack_patches
from microscope.computations_grid.curvature import (
    _scalar_curvature_in_patches,
    scalar_curvature_batch,
)
from microscope.cyclic_dimensions import get_difference_intervals
from microscope.manifold_examples.sampling_grid import (
    sample_ellipsoid_on_grid,
    sample_hyperboloid_on_grid,
)

In [2]:
def scalar_curvature(
    features_on_grid: np.ndarray,
    difference_intervals: list[float],
    cyclic_dimensions: list[int],
    patch_sizes: list[int],
    normalize: bool = False,
    device: str | torch.device | None = None
) -> np.ndarray:
    """Estimate scalar curvature of a NumPy grid in patches.

    Args:
        features_on_grid: Floating array of shape (s1, ..., sd, features).
        difference_intervals: One coordinate spacing per grid axis.
        cyclic_dimensions: Zero-based periodic grid-axis indices.
        patch_sizes: One patch size per grid axis, each greater than six.
        normalize: Divide scalar curvature by d * (d - 1); requires d >= 2.
            This does not rescale the manifold to unit volume.
        device: Torch device; None selects CUDA when available, otherwise CPU.

    Returns:
        A NumPy array of shape (s1_out, ..., sd_out), where s_i_out = s_i
        for periodic axes and s_i - 6 otherwise: one scalar curvature per point.
    """
    if device is None:
        device = "cuda:0" if torch.cuda.is_available() else "cpu"

    dims = len(patch_sizes)
    overlaps = dims * [6]

    patches = extract_patches(
        features_on_grid,
        patch_sizes,
        overlaps,
        cyclic_dimensions=cyclic_dimensions
    )

    volume_patches = []
    # Flattening the strided patch grid copies every overlapping patch.
    for index in np.ndindex(*patches.shape[:dims]):
        patch = patches[index]
        points_pt_patch = torch.from_numpy(patch).to(device)

        volume_pred = scalar_curvature_batch(
            points_pt_patch,
            difference_intervals=difference_intervals,
            normalize=normalize
        ).cpu().detach().numpy()
        volume_patches.append(volume_pred)

    volume_patches = np.stack(volume_patches, axis=0).reshape(
        *patches.shape[:dims],
        *list(np.array(patches.shape[dims:2 * dims]) - np.array(overlaps)),
        *patches.shape[2 * dims:-1],
    )

    curvature_array = stack_patches(volume_patches, n_features=0)

    # Truncate any padded zeros introduced in the patch extraction.
    expected_array_shape = tuple(
        slice(
            0,
            s if i in cyclic_dimensions else s - 6
        )
        for i, s in enumerate(features_on_grid.shape[:-1])
    )

    return curvature_array[expected_array_shape]

In [3]:
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
RUN_SLOW_3D = False
RUN_GIANT_TOTAL_GRID = False
APPLY_RANDOM_ISOMETRY = False  # Same surfaces; enable to match the tests' orientation.
DTYPES = (np.float64,)          # Add np.float32 for a separate dtype check.
NORMALIZE_VALUES = (False,)     # Add True to check normalized outputs too.

cases = [
    dict(name="2D sphere / regular", kind="ellipsoid", axes=[2, 2, 2],
         sizes=[1006, 1000], ambient=3, patches=[56, 56], fraction=.8),
    dict(name="2D ellipsoid / regular", kind="ellipsoid", axes=[1, 1.5, 3],
         sizes=[1006, 1000], ambient=3, patches=[56, 56], fraction=.8),
    dict(name="2D sphere / irregular", kind="ellipsoid", axes=[2, 2, 2],
         sizes=[1223, 1344], ambient=3, patches=[34, 42], fraction=.8),
    dict(name="2D ellipsoid / irregular", kind="ellipsoid", axes=[1, 1.5, 3],
         sizes=[1223, 1344], ambient=3, patches=[34, 42], fraction=.8),
    dict(name="2D hyperboloid", kind="hyperboloid", axes=[5, 1, 3],
         sizes=[1006, 1000], ambient=3, patches=[56, 56]),
]
if RUN_SLOW_3D:
    for ambient in (4, 100, 4096):
        cases.append(dict(name=f"3D ellipsoid / ambient {ambient}", kind="ellipsoid",
                          axes=[1, 1.5, 3, 1.8],
                          sizes=[56, 56, 50] if ambient > 100 else [106, 106, 100],
                          ambient=ambient, patches=[16, 16, 16], fraction=.8))
    for ambient in (4, 100):
        cases.append(dict(name=f"3D hyperboloid / ambient {ambient}", kind="hyperboloid",
                          axes=[5, 1, 3, 1], sizes=[156, 156, 150],
                          ambient=ambient, patches=[56, 56, 56]))
if RUN_GIANT_TOTAL_GRID:
    for name, axes in (("2D sphere / total grid", [2, 2, 2]),
                       ("2D ellipsoid / total grid", [1, 1.5, 3])):
        cases.append(dict(name=name, kind="ellipsoid", axes=axes,
                          sizes=[10223, 10344], ambient=3,
                          patches=[1034, 1042], fraction=.9999))


def sample_case(case, dtype):
    sizes = case["sizes"]
    cyclic = [len(sizes) - 1]
    if case["kind"] == "ellipsoid":
        points, _, _, _ = sample_ellipsoid_on_grid(
            semi_axes=case["axes"], n_samples_per_dim=sizes,
            ambient_space_dim=case["ambient"],
            fraction_of_angles=case["fraction"],
            apply_random_isometry=APPLY_RANDOM_ISOMETRY, seed=123,
        )
        ranges = [(2 * case["fraction"] - 1) * np.pi] * (len(sizes) - 1) + [2 * np.pi]
    else:
        points, _, _, _ = sample_hyperboloid_on_grid(
            semi_axes=case["axes"], n_samples_per_dim=sizes,
            ambient_space_dim=case["ambient"],
            apply_random_isometry=APPLY_RANDOM_ISOMETRY, seed=123,
        )
        ranges = [2.] * (len(sizes) - 1) + [2 * np.pi]
    return points.astype(dtype, copy=False), get_difference_intervals(sizes, ranges, cyclic), cyclic

In [4]:
def compare_arrays(original, current):
    report = dict(shape_equal=original.shape == current.shape,
                  dtype_equal=original.dtype == current.dtype,
                  byte_differences=None, value_differences=None,
                  first_difference=None)
    if not (report["shape_equal"] and report["dtype_equal"]):
        return report

    byte_differences = 0
    value_differences = 0
    offset = 0
    itemsize = original.dtype.itemsize
    iterator = np.nditer([original, current],
                         flags=["external_loop", "buffered"],
                         op_flags=[["readonly"], ["readonly"]],
                         order="C", buffersize=262144)
    for old_chunk, new_chunk in iterator:
        old_bytes = old_chunk.view(np.uint8).reshape(-1, itemsize)
        new_bytes = new_chunk.view(np.uint8).reshape(-1, itemsize)
        different = np.any(old_bytes != new_bytes, axis=1)
        count = int(np.count_nonzero(different))
        byte_differences += count
        same_value = (old_chunk == new_chunk) | (np.isnan(old_chunk) & np.isnan(new_chunk))
        value_differences += int(np.count_nonzero(~same_value))
        if count and report["first_difference"] is None:
            flat_index = offset + int(np.flatnonzero(different)[0])
            index = tuple(int(i) for i in np.unravel_index(flat_index, original.shape))
            report["first_difference"] = dict(index=index,
                                               original=original[index].item(),
                                               current=current[index].item())
        offset += old_chunk.size
    assert offset == original.size
    report["byte_differences"] = byte_differences
    report["value_differences"] = value_differences
    return report


def compare_case(case, dtype=np.float64, normalize=False, device=DEVICE):
    points, intervals, cyclic = sample_case(case, dtype)
    original = scalar_curvature(
        features_on_grid=points,
        difference_intervals=intervals,
        cyclic_dimensions=cyclic,
        patch_sizes=case["patches"],
        normalize=normalize,
        device=device
    )
    current = _scalar_curvature_in_patches(
        features_on_grid=points,
        difference_intervals=intervals,
        cyclic_dimensions=cyclic,
        patch_sizes=case["patches"],
        normalize=normalize,
        device=device
    )
    report = compare_arrays(original, current)
    return dict(case=case["name"], input_shape=points.shape,
                dtype=str(dtype), normalize=normalize, device=str(device),
                original_shape=original.shape, current_shape=current.shape,
                original_dtype=str(original.dtype), current_dtype=str(current.dtype),
                **report)

In [5]:
results = []
for case in cases:
    for dtype in DTYPES:
        for normalize in NORMALIZE_VALUES:
            row = compare_case(case, dtype=dtype, normalize=normalize)
            results.append(row)
            status = "EXACT" if (row["shape_equal"] and row["dtype_equal"]
                                 and row["byte_differences"] == 0) else "DIFFERENT"
            print(f"{status:9s} {row['case']:32s} {row['input_shape']!s:25s} "
                  f"{row['dtype']:22s} normalize={normalize} "
                  f"byte-different elements={row['byte_differences']}")
            if row["first_difference"] is not None:
                print("  first difference:", row["first_difference"])

assert all(row["shape_equal"] and row["dtype_equal"]
           and row["byte_differences"] == 0 for row in results)
print(f"All {len(results)} cases match element by element and byte for byte.")

EXACT     2D sphere / regular              (1006, 1000, 3)           <class 'numpy.float64'> normalize=False byte-different elements=0
EXACT     2D ellipsoid / regular           (1006, 1000, 3)           <class 'numpy.float64'> normalize=False byte-different elements=0
EXACT     2D sphere / irregular            (1223, 1344, 3)           <class 'numpy.float64'> normalize=False byte-different elements=0
EXACT     2D ellipsoid / irregular         (1223, 1344, 3)           <class 'numpy.float64'> normalize=False byte-different elements=0
EXACT     2D hyperboloid                   (1006, 1000, 3)           <class 'numpy.float64'> normalize=False byte-different elements=0
All 5 cases match element by element and byte for byte.
